# DataSays Olist Business Analysis Suite v2

这个 Notebook 用于逐题运行 24 题模型基线。每次调用 `await run_next_case()` 只运行一道题，并展示答案、结构化计划、事实评分、业务要点、澄清、记忆和可视化检查。

- 不需要单独启动 FastAPI，Notebook 使用 in-process ASGI 调用真实 `/api/query`。
- 仍会经过 LangGraph、模型调用、Python 执行、验证、修复和持久化服务。
- 每题结果按 benchmark 版本自动保存到 `server/evals/results/`，重新打开后会跳过当前版本已保存的题目。
- 请在 `server/.env` 中配置 `OPENROUTER_API_KEY`，并选择项目的 `datasays` Python 环境。
- 模型调用会产生费用。默认不会自动运行全部 24 题。


In [ ]:
# import sys
# print(sys.executable)

In [ ]:
# import fastapi
# print(fastapi.__version__)

In [ ]:
import importlib
import json
import os
import re
import sys
import traceback
from pathlib import Path

import pandas as pd
try:
    from IPython.display import Markdown, display
except ImportError:
    class Markdown(str):
        pass
    def display(value):
        print(value)

def find_server_dir(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'app').is_dir() and (candidate / 'evals').is_dir():
            return candidate
        if (candidate / 'server' / 'app').is_dir():
            return candidate / 'server'
    raise RuntimeError('找不到 DataSays server 目录，请从项目目录打开此 Notebook。')

SERVER_DIR = find_server_dir(Path.cwd().resolve())
os.chdir(SERVER_DIR)
if str(SERVER_DIR) not in sys.path:
    sys.path.insert(0, str(SERVER_DIR))

from dotenv import load_dotenv
load_dotenv(SERVER_DIR / '.env')

run_eval_module = importlib.import_module('evals.run_eval')
importlib.reload(run_eval_module)
run_business_eval_module = importlib.import_module('evals.run_business_eval')
run_business_eval_module = importlib.reload(run_business_eval_module)
run_business_eval = run_business_eval_module.run_business_eval

CASES_PATH = SERVER_DIR / 'evals' / 'business_benchmark_cases.json'
RESULTS_DIR = SERVER_DIR / 'evals' / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# 修改为你要测试的 OpenRouter model ID。建议一个模型完整跑完后再换模型。
MODEL = 'qwen/qwen3.6-flash'

CONFIG = json.loads(CASES_PATH.read_text(encoding='utf-8'))
CASES = CONFIG['cases']
CASE_BY_ID = {case['id']: case for case in CASES}
CASE_IDS = [case['id'] for case in CASES]
MODEL_KEY = re.sub(r'[^A-Za-z0-9._-]+', '-', MODEL).strip('-').lower()
BENCHMARK_KEY = re.sub(r'[^A-Za-z0-9._-]+', '-', CONFIG.get('version', 'unknown')).strip('-').lower()

if not os.getenv('OPENROUTER_API_KEY'):
    raise RuntimeError('未找到 OPENROUTER_API_KEY，请检查 server/.env。')

print(f'Server: {SERVER_DIR}')
print(f'Model: {MODEL}')
print(f'Benchmark: {CONFIG.get("benchmark_name")} {CONFIG.get("version")}')
print(f'Cases: {len(CASES)}')
print(f'Results: {RESULTS_DIR}')


## 题目列表与当前进度

重新运行下一个代码单元可以刷新状态。`saved = True` 只表示结果文件已经保存，不代表题目通过。


In [ ]:
def result_path(case_id: str) -> Path:
    return RESULTS_DIR / f'v2-{BENCHMARK_KEY}-{MODEL_KEY}-{case_id}.json'

def saved_report(case_id: str):
    path = result_path(case_id)
    if not path.exists():
        return None
    return json.loads(path.read_text(encoding='utf-8'))

rows = []
for index, case in enumerate(CASES, start=1):
    report = saved_report(case['id'])
    rows.append({
        '序号': index,
        'case_id': case['id'],
        '类别': case['category'],
        '数据表': ', '.join(case.get('datasets', [])),
        '用户需求': case.get('user_need'),
        'saved': report is not None,
        'passed': report.get('passed') == 1 if report and 'passed' in report else None,
    })
display(pd.DataFrame(rows))


## 运行与展示函数

运行本单元只会定义函数，不会调用模型。底层 Runner 会为多轮题创建真实会话，并在结束后清理评测上传文件与临时会话。


In [ ]:
def display_case_definition(case_id: str) -> None:
    case = CASE_BY_ID[case_id]
    dataset_names = ', '.join(case.get('datasets', []))
    display(Markdown(f"### {case_id}\n\n**类别：** {case['category']}  \n**使用数据表：** {dataset_names}  \n**用户需求：** {case.get('user_need', '')}  \n**能力：** {case.get('capability', '')}"))
    turns = case.get('turns') or [{'question': case['question'], 'expected': case['expected']}]
    for index, turn in enumerate(turns, start=1):
        display(Markdown(f"**第 {index} 轮问题：** {turn['question']}"))
        expected = turn['expected']
        fact_rows = [
            {'fact_id': fact['id'], 'expected': fact['value'], 'tolerance': fact.get('tolerance')}
            for fact in expected.get('facts', [])
        ]
        if fact_rows:
            display(pd.DataFrame(fact_rows))
        print('业务要点:', expected.get('required_term_groups', []))
        print('预期澄清:', expected.get('clarification', False), '| 预期使用记忆:', expected.get('memory_used'))

def display_turn(turn: dict) -> None:
    status = '通过' if turn.get('passed') else '失败'
    display(Markdown(f"#### 第 {turn['turn']} 轮：{status}\n\n**问题：** {turn['question']}\n\n**最终答案：**\n\n{turn.get('answer') or '无'}"))

    summary = pd.DataFrame([{
        'passed': turn.get('passed'),
        'latency_seconds': turn.get('latency_seconds'),
        'fact_recall': turn.get('fact_recall'),
        'term_coverage': turn.get('term_coverage'),
        'clarification': turn.get('clarification_observed'),
        'memory_used': turn.get('memory_observed'),
        'validation_passed': turn.get('validation_passed'),
        'repair_attempts': turn.get('repair_attempts'),
    }])
    display(summary)

    fact_results = turn.get('fact_results') or []
    if fact_results:
        display(Markdown('**事实评分**'))
        display(pd.DataFrame(fact_results))

    term_groups = turn.get('term_groups') or []
    if term_groups:
        display(Markdown('**业务要点评分**'))
        display(pd.DataFrame(term_groups))

    display(Markdown('**结构化分析计划**'))
    display(pd.json_normalize(turn.get('plan') or {}, sep='.').T.rename(columns={0: 'value'}))

    result = turn.get('structured_result')
    if result:
        display(Markdown('**结构化结果**'))
        display(pd.json_normalize(result, sep='.').T.rename(columns={0: 'value'}))

    validation = turn.get('validation_report')
    if validation:
        display(Markdown('**验证报告**'))
        checks = validation.get('checks') or []
        if checks:
            display(pd.DataFrame(checks))
        else:
            print(validation)

def display_report(report: dict) -> None:
    if report.get('runner_error'):
        display(Markdown(f"### Runner Error\n\n```text\n{report['runner_error']}\n```"))
        print(report.get('traceback', ''))
        return
    display(pd.DataFrame([{
        'case_passed': report.get('passed'),
        'case_failed': report.get('failed'),
        'fact_recall': report.get('fact_recall'),
        'business_term_coverage': report.get('business_term_coverage'),
        'clarification_accuracy': report.get('clarification_accuracy'),
        'memory_accuracy': report.get('memory_accuracy'),
        'visualization_coverage': report.get('visualization_coverage'),
        'average_latency_seconds': report.get('average_latency_seconds'),
    }]))
    for case_result in report.get('results', []):
        for turn in case_result.get('turns', []):
            display_turn(turn)

async def run_case(case_id: str, overwrite: bool = False) -> dict:
    if case_id not in CASE_BY_ID:
        raise KeyError(f'未知 case_id: {case_id}')
    path = result_path(case_id)
    if path.exists() and not overwrite:
        print(f'读取已保存结果：{path.name}')
        report = json.loads(path.read_text(encoding='utf-8'))
        display_report(report)
        return report

    display_case_definition(case_id)
    print(f'开始运行 {case_id}，模型：{MODEL}。复杂题可能需要数分钟……', flush=True)
    try:
        report = await run_business_eval(
            api_base='http://127.0.0.1:8000',
            cases_path=CASES_PATH,
            case_ids=[case_id],
            model_override=MODEL,
            local_files=True,
            in_process_api=True,
        )
    except Exception as exc:
        report = {
            'benchmark_name': CONFIG.get('benchmark_name'),
            'model': MODEL,
            'case_id': case_id,
            'runner_error': str(exc),
            'traceback': traceback.format_exc(),
        }
    path.write_text(json.dumps(report, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')
    print(f'结果已保存：{path}')
    display_report(report)
    return report

def next_case_id():
    for case_id in CASE_IDS:
        report = saved_report(case_id)
        if report is None or report.get('runner_error'):
            return case_id
    return None

async def run_next_case() -> dict | None:
    case_id = next_case_id()
    if case_id is None:
        print('当前模型的 24 题都已有保存结果。')
        return None
    previous = saved_report(case_id)
    return await run_case(case_id, overwrite=bool(previous and previous.get('runner_error')))

print('函数已准备。下一题：', next_case_id())


## 每次运行下一题

反复运行下面这个单元即可。Runner Error 会在下次自动覆盖重试；也可用 `await run_case('case_id', overwrite=True)` 指定覆盖重跑。


In [ ]:
current_report = await run_next_case()


## 指定题目重跑（可选）

取消下一行注释并替换 case ID。`overwrite=True` 会覆盖该模型这道题的旧结果。


In [ ]:
# current_report = await run_case('profit_metric_clarification', overwrite=True)


## 汇总当前模型的已保存结果

可以在任意阶段运行。Runner Error 与正常失败分开统计。完整基线只有在 24 题都保存且人工抽查后才适合对外使用。


In [ ]:
summary_rows = []
for index, case_id in enumerate(CASE_IDS, start=1):
    report = saved_report(case_id)
    if report is None:
        summary_rows.append({'序号': index, 'case_id': case_id, 'status': 'not_run'})
        continue
    if report.get('runner_error'):
        summary_rows.append({
            '序号': index, 'case_id': case_id, 'status': 'runner_error',
            'error': report.get('runner_error'),
        })
        continue
    summary_rows.append({
        '序号': index,
        'case_id': case_id,
        'status': 'passed' if report.get('passed') == 1 else 'failed',
        'fact_recall': report.get('fact_recall'),
        'term_coverage': report.get('business_term_coverage'),
        'clarification': report.get('clarification_accuracy'),
        'memory': report.get('memory_accuracy'),
        'visualization': report.get('visualization_coverage'),
        'latency_seconds': report.get('average_latency_seconds'),
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)
display(summary_df['status'].value_counts(dropna=False).rename_axis('status').to_frame('cases'))
